# Image RAS Array Helper Test

This notebook tests the new image-side temporary RAS array helpers while keeping the in-memory `MedicalImage` object strictly canonical `LPS+`.

## What this notebook verifies

1. We load a real `RAS+` NIfTI file with `image_io.read_nifti(path)`. This returns a canonical `MedicalImage` whose internal array is reoriented to `LPS+`.
2. We convert that canonical image into a temporary `RAS+` NumPy array with `image_io.medical_image_to_ras_array(image)`.
3. We load the same NIfTI file with `image_io.read_nifti_plain(path)`, which returns the raw on-disk NIfTI array without reorientation.
4. We compare the two `RAS+` arrays and verify they are identical.
5. We test the inverse helper `image_io.medical_image_from_ras_array_like(array_ras, reference_image)` by converting the temporary `RAS+` array back into a new canonical `LPS+` `MedicalImage` that reuses the reference image geometry.
6. We verify that the reconstructed `MedicalImage` matches the original canonical object.

## Why this is useful

The project keeps `MedicalImage` objects clean and easy to debug by storing them only in canonical `LPS+`. The new helpers let us temporarily interoperate with code that expects canonical `RAS+` arrays, without introducing mixed-orientation objects into the main API.


In [1]:
from pathlib import Path
import sys

import numpy as np

# Make the project importable even when the notebook is launched from this folder.
PROJECT_ROOT = Path('/homebase')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from code_sync.myMedIADIRLab.data_io.image import image_io

PATH_IMAGE = Path('/homebase/code_sync/myMedIADIRLab/data_io/unittest_image/data_local/LUMIRMRI_3454_0000.nii.gz')
print(f'Input file exists: {PATH_IMAGE.exists()}')
print(f'Input file: {PATH_IMAGE}')


Input file exists: True
Input file: /homebase/code_sync/myMedIADIRLab/data_io/unittest_image/data_local/LUMIRMRI_3454_0000.nii.gz


## Step 1: Load the image in the two ways we want to compare

In this step we create two representations of the same file:

- `image_lps`: canonical `MedicalImage` loaded with `read_nifti`. Its internal array is `LPS+`.
- `array_ras_plain`: raw array loaded with `read_nifti_plain`. This is the on-disk NIfTI array, which for this file is expected to already be `RAS+`.

We also print the canonical image shape and affine so it is clear that the object stores geometry in canonical `LPS+` space.


In [2]:
image_lps = image_io.read_nifti(PATH_IMAGE)
array_ras_plain, affine_ras_plain = image_io.read_nifti_plain(PATH_IMAGE)

print('Canonical MedicalImage information')
print('  shape           :', image_lps.shape)
print('  spatial_ndim    :', image_lps.spatial_ndim)
print('  affine_lps      :')
print(image_lps.affine_lps)
print()
print('Plain NIfTI information')
print('  raw array shape :', array_ras_plain.shape)
print('  raw affine_ras  :')
print(affine_ras_plain)


Canonical MedicalImage information
  shape           : (160, 224, 192)
  spatial_ndim    : 3
  affine_lps      :
[[   1.    0.    0. -159.]
 [   0.    1.    0. -223.]
 [   0.    0.    1.    0.]
 [   0.    0.    0.    1.]]

Plain NIfTI information
  raw array shape : (160, 224, 192)
  raw affine_ras  :
[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]


## Step 2: Convert the canonical `MedicalImage` back to a temporary `RAS+` array

`medical_image_to_ras_array(image_lps)` should undo the internal `LPS+` storage orientation and give us a plain `RAS+` array that matches the raw on-disk NIfTI array.

This is the key interoperability test for the new helper.


In [3]:
array_ras_from_image = image_io.medical_image_to_ras_array(image_lps)

print('RAS array comparison')
print('  from MedicalImage shape :', array_ras_from_image.shape)
print('  plain NIfTI shape       :', array_ras_plain.shape)
print('  dtypes                  :', array_ras_from_image.dtype, array_ras_plain.dtype)
print('  np.array_equal          :', np.array_equal(array_ras_from_image, array_ras_plain))
print('  np.allclose             :', np.allclose(array_ras_from_image, array_ras_plain))

assert array_ras_from_image.shape == array_ras_plain.shape
assert np.array_equal(array_ras_from_image, array_ras_plain)


RAS array comparison
  from MedicalImage shape : (160, 224, 192)
  plain NIfTI shape       : (160, 224, 192)
  dtypes                  : float64 float64
  np.array_equal          : True
  np.allclose             : True


## Step 3: Round-trip the temporary `RAS+` array back into a new canonical `LPS+` `MedicalImage`

Now we test the inverse helper.

`medical_image_from_ras_array_like(array_ras, reference_image)` should:

- interpret `array_ras` as canonical `RAS+`,
- convert it back into canonical `LPS+`,
- reuse the reference image affine and metadata layout,
- and return a new `MedicalImage` object.

If this helper is correct, the reconstructed image should match the original canonical image exactly in array content and geometry.


In [4]:
image_lps_roundtrip = image_io.medical_image_from_ras_array_like(
    array_ras=array_ras_from_image,
    reference_image=image_lps,
)

array_equal_roundtrip = np.array_equal(image_lps_roundtrip.array, image_lps.array)
affine_equal_roundtrip = np.allclose(image_lps_roundtrip.affine_lps, image_lps.affine_lps)
shape_equal_roundtrip = image_lps_roundtrip.shape == image_lps.shape
spatial_ndim_equal_roundtrip = image_lps_roundtrip.spatial_ndim == image_lps.spatial_ndim

print('Round-trip LPS object comparison')
print('  array equal      :', array_equal_roundtrip)
print('  affine equal     :', affine_equal_roundtrip)
print('  shape equal      :', shape_equal_roundtrip)
print('  spatial_ndim eq  :', spatial_ndim_equal_roundtrip)
print('  metadata key set :', 'medical_image_from_ras_array_like' in image_lps_roundtrip.metadata)

assert array_equal_roundtrip
assert affine_equal_roundtrip
assert shape_equal_roundtrip
assert spatial_ndim_equal_roundtrip


Round-trip LPS object comparison
  array equal      : True
  affine equal     : True
  shape equal      : True
  spatial_ndim eq  : True
  metadata key set : True


## Summary

This notebook confirms two important behaviors:

1. `medical_image_to_ras_array` correctly converts a canonical `LPS+` `MedicalImage` into the same `RAS+` array that is stored on disk in this NIfTI file.
2. `medical_image_from_ras_array_like` correctly converts that temporary `RAS+` array back into a new canonical `LPS+` `MedicalImage` while preserving the reference geometry.

This gives us a clean workflow for temporary `RAS+` array interoperability without changing the canonical internal `MedicalImage` convention.
